# MM4250 stuck check

Steps the switch through ALL_OPEN, internal short, ALL_OPEN, internal load, ALL_OPEN, then RF1, ALL_OPEN, RF2, ALL_OPEN ... RF6, ALL_OPEN, and saves one raw `.s2p` per position plus `positions.csv` (time, fridge temperature) in `Sweeps/stuck_check/<date>_<time>_<serial>_<temp>/` beside this notebook. Nothing goes into a database.

- Run the cells top to bottom, once each.
- **First close `mm4250_sweeps.ipynb`** (its Close cell, or restart its kernel): only one notebook can hold the switch.
- VNA port 1 on fridge line RF6, port 2 on line RF2. **The LNA must be on** (4 K bias when cold), or nothing comes back and every position looks the same.

In [ ]:
import sys, csv
from datetime import datetime
from pathlib import Path
import numpy as np

try:
    here = Path(_dh[0])      # this notebook's folder (Jupyter)
except NameError:
    here = Path.cwd()
root = here
while not (root / "drivers").is_dir() and root != root.parent:
    root = root.parent       # the repo with drivers/ in it
sys.path[:0] = [str(here), str(root)]

from drivers.KeysightVNA_driver import KeysightP5004B
from drivers.MM4250_QCodes_driver import MM4250
from vna_measure import setup_sweep, measure_2port   # moves the switch with the VNA source off
from sweep_db import _leave_switch_open
from read_db import save_touchstone

**Settings.** Nothing to change normally: the temperature in the folder name comes from the fridge. If the fridge isn't connected, type it in `TEMP_LABEL`.

In [ ]:
TEMP_LABEL = None                   # None = from the fridge (mixing plate); or type one, e.g. "60K"
SERIAL = "SN0077"
CHANNELS = [1, 2, 3, 4, 5, 6]       # switch channels to check
SWEEP = dict(start=510e6, stop=610e6, points=2001, if_bandwidth=1e3, power=-20)   # circulator band, -20 dBm

**Connect.** Connecting the switch sets ALL_OPEN. The fridge is only for temperatures; if it doesn't connect (no VPN), the check still runs.

In [ ]:
from qcodes.instrument import Instrument
Instrument.close_all()              # so re-running this cell is safe

ksvna = KeysightP5004B("ksvna", "TCPIP0::QTSF-Measurement::hislip_PXI0_CHASSIS1_SLOT1_INDEX0::INSTR")
switch = MM4250("switch")

try:
    from maybell_qcodes import MaybellFridge
    fridge = MaybellFridge("fridge", host="10.179.200.168")
except Exception as e:
    fridge = None
    print(f"No fridge temperatures ({type(e).__name__}: {e}). The check still runs.")

def mxp_kelvin():
    """Mixing plate temperature in K, or None if there's no reading."""
    try:
        return fridge.mxp.temperature() if fridge is not None else None
    except Exception:
        return None

**Run.** About a minute. Reading the printout (compare neighbouring rows, not fixed numbers):

- Every ALL_OPEN row small, every other row clearly bigger: **all good**.
- A position about as small as the ALL_OPEN rows: it **didn't close** (stuck open).
- An ALL_OPEN row as big as the position before it: that position **didn't open** (stuck closed).

If anything is stuck, stop: don't keep switching it cold. Run the Close cell and tell Charlie. The switch is always left ALL_OPEN at the end, even if this cell fails.

In [ ]:
seq = ["ALL_OPEN", "INTERNAL_SHORT", "ALL_OPEN", "INTERNAL_LOAD", "ALL_OPEN"]
for ch in CHANNELS:
    seq += [ch, "ALL_OPEN"]

t0 = mxp_kelvin()
label = TEMP_LABEL or (f"{t0:.0f}K" if t0 is not None and t0 >= 1 else f"{t0 * 1e3:.0f}mK" if t0 is not None else "noT")
out = here / "Sweeps" / "stuck_check" / f"{datetime.now():%Y%m%d_%H%M%S}_{SERIAL}_{label}"
setup_sweep(vna=ksvna, **SWEEP)
ksvna.write("SENS:CORR:CSET:DEAC")  # raw data: VNA calibration off

rows, ref = [], None
try:
    for i, pos in enumerate(seq, 1):
        name = f"RF{pos}" if isinstance(pos, int) else pos
        temp_k = mxp_kelvin()
        if isinstance(pos, int):
            freq, data = measure_2port(pos, vna=ksvna, switch=switch)
        else:
            freq, data = measure_2port(state=pos, vna=ksvna, switch=switch)
        path = out / f"{i:02d}_{name}.s2p"
        save_touchstone(freq, data, path)
        if ref is None:
            ref = data["S21"]       # the first ALL_OPEN
        diff = float(np.median(np.abs(data["S21"] - ref)))
        rows.append([i, name, path.name, datetime.now().isoformat(timespec="seconds"), temp_k, diff])
finally:
    _leave_switch_open(switch, ksvna)   # ALL_OPEN, even after an error
    if rows:
        with open(out / "positions.csv", "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["step", "position", "file", "time", "mxp_K", "median_abs_S21_minus_first_open"])
            w.writerows(rows)

print(f"\nSaved {len(rows)} files to {out}\n")
for i, name, _, _, temp_k, diff in rows:
    t = "no reading" if temp_k is None else f"{temp_k * 1e3:.1f} mK" if temp_k < 1 else f"{temp_k:.2f} K"
    print(f"{i:02d}  {name:>14}   difference from first ALL_OPEN: {diff:.3g}   ({t})")

**Close.** Run when done, before anyone else uses the switch or VNA.

In [ ]:
ksvna.output(False)
switch.open_all()
ksvna.close()
switch.close()
if fridge is not None:
    fridge.close()